# Oppitunti 12 - Keskusteluhistorian tiivistäminen agentin luonnosmuistilla

Tämä muistikirja esittelee, miten hallita kontekstia pitkissä keskusteluissa Microsoft Agent Frameworkin avulla. Kun keskustelut kasvavat, token-määrä kasvaa — lopulta ylittäen mallin konteksti-ikkunan. Ratkaisemme tämän **kontekstin tiivistämisellä** ja **agentin luonnosmuistilla**, joka tarjoaa pysyvän muistin.

## Mitä opit:
1. **Miksi kontekstinhallinta on tärkeää**: Token-rajat ja konteksti-ikkunoiden ymmärtäminen
2. **Kontekstin tuntevat agentit**: Agenttien rakentaminen, jotka hallitsevat omaa keskustelukontekstiaan
3. **Kontekstin tiivistämismalli**: Työkalujen käyttö keskusteluhistorian tiivistämiseen
4. **Agentin luonnosmuisti**: Pysyvä muisti, joka säilyy kontekstin tiivistämisestä huolimatta

## Edellytykset:
- Azure OpenAI -ympäristön asetus ympäristömuuttujilla
- Peruskäsitys agenteista aiemmista oppitunneista


## Asennus


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Miksi kontekstinhallinta on tärkeää

Jokaisella LLM:llä on rajallinen **kontekstikkuna** — maksimimäärä tokeneita, joita se voi käsitellä yhdessä pyynnössä. Monikierroksisen keskustelun edetessä:

- **Tokenien määrä kasvaa lineaarisesti** jokaisen käyttäjän viestin ja avustajan vastauksen myötä.
- **Kehote-tokenit muodostavat kustannukset** koska koko historia lähetetään uudelleen jokaisella kierroksella.
- Lopulta keskustelu **ylittää kontekstikkunan** ja malli joko katkaisee tai antaa virheen.

### Strategiat kontekstinhallintaan

| Strategia | Miten se toimii | Kompromissi |
|---|---|---|
| **Katkaisu** | Pudota vanhimmat viestit pois | Menettää alkuperäisen kontekstin |
| **Tiivistäminen** | Tiivistetään vanhemmat viestit yhteenvedoksi | Jotain yksityiskohtia katoaa, mutta avainkohdat säilyvät |
| **Muistilehtiö / Ulkoinen muisti** | Tallenna avaintiedot keskustelun ulkopuolelle | Vaatii työkalukutsuja, mutta säilyy kaikissa tiivistyksissä |

Tässä muistikirjassa yhdistämme **tiivistämisen** ja **muistilehtiötyökalun**, jotta agentti voi ylläpitää jatkuvuutta, vaikka keskusteluhistoria tiivistetään.


## Kontekstia tuntevan agentin luominen


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Pitkän keskustelun simulointi

Käydään läpi monivuoroinen keskustelu nähdäksesi, miten konteksti kertyy. Agentin tulisi säilyttää keskeiset tiedot (mieltymykset, budjetti, matkustusajat) vuorojen välillä ja osoittaa jatkuvuutta.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Huomaa, kuinka agentti säilyttää kontekstin aiemmista käännöksistä — se tietää Japanista, sushista, temppeleistä, valokuvauksesta, 3000 dollarin budjetista, yksin matkustamisesta ja huhtikuun aikataulusta. Lyhyessä keskustelussa tämä toimii hyvin, mutta keskustelun kasvaessa koko historia tulee kalliiksi lähettää uudelleen.

Jatketaan keskustelua lisää käännöksillä nähdäksesi kontekstin kertyminen:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Kontekstin tiivistysmalli

Keskustelun kasvaessa voimme käyttää **tiivisty työkalua** tiivistämään kertynyttä kontekstia tiiviiseen muotoon. Agentti kutsuu tätä työkalua tallentaakseen keskeiset mieltymykset niin, että vaikka vanhat viestit poistettaisiin, oleellinen tieto säilyy.

Tämä malli on rakennuspalikka monimutkaisempaan historian vähentämiseen:
1. Agentti tunnistaa keskustelusta keskeiset faktat
2. Se kutsuu tiivisty työkalua tallentaakseen ne
3. Vanhemmat viestit voidaan turvallisesti poistaa, koska tiivistelmä tallentaa olennaisen

Alla määrittelemme `summarize_preferences` työkalun, jota agentti voi kutsua tallentaakseen kompaktin yhteenvedon oppimastaan.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Yhteenveto

Tässä oppitunnissa opit hallitsemaan kontekstia pitkissä agenttikeskusteluissa Microsoft Agent Frameworkia käyttäen:

### Keskeiset käsitteet
- **Konteksti-ikkunat ovat rajallisia** — jokainen keskusteluhistoriassa oleva token maksaa ja lasketaan rajaan mukaan.
- **Yhteenvetotyökalut** antavat agentin tiivistää kertyneen kontekstin kompakteiksi yhteenvetoiksi, mikä vähentää tokenien käyttöä samalla, kun olennaiset tiedot säilyvät.
- **Agentin muistiinpanot** tarjoavat pysyvän ulkoisen muistin, joka säilyy keskustelun lyhennyksistä huolimatta.

### Mitä Rakensit
- **Kontekstia ymmärtävä agentti**, joka ylläpitää jatkuvuutta monivaiheisissa keskusteluissa
- **Yhteenvedon työkalun** (`summarize_preferences`), joka tallentaa käyttäjän keskeiset tiedot kompaktissa muodossa
- **Monivaiheisen keskustelun**, joka havainnollistaa kontekstin säilyttämistä ja muutosten käsittelyä

### Käytännön sovellukset
- **Asiakaspalvelubotit**: Muista mieltymykset pitkien tukisessioiden aikana
- **Henkilökohtaiset avustajat**: Seuraa käynnissä olevia projekteja ilman, että konteksti pitää selittää uudelleen
- **Opetusavustajat**: Ylläpidä opiskelijan edistymistä monien vuorovaikutusten aikana

### Seuraavat askeleet
- Toteuta täydellinen muistiinpanotyökalu tiedostopohjaisella säilytyksellä
- Lisää automaattinen historian lyhennys yhteenvedon jälkeen
- Yhdistä vektoritietokantoihin semanttista muistin hakua varten
- Rakenna agenteista keskustelut, jotka voivat jatkaa vuorovaikutuksia päivien päästä täyden kontekstin kanssa


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
